In [ ]:
!pip install langchain

### First LangChain Code
- Install langchain-ollama

In [ ]:
!pip install langchain-ollama

### First chat with Ollama using Langchain

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

template = PromptTemplate.from_template("What is the capital of {countryName}?")
prompt = template.invoke({"countryName": "France"})


chatTemplate = ChatPromptTemplate.from_messages()

print(prompt)

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )

res = llm_model.invoke(prompt)
print(res)



text='What is the capital of France?'
content='The capital of France is Paris.' additional_kwargs={} response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-03-31T14:38:04.67813882Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2231798628, 'load_duration': None, 'prompt_eval_count': 15, 'prompt_eval_duration': None, 'eval_count': 8, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'} id='lc_run--019d4454-7c61-7813-a090-cb022bf0f38e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23}


### Different Types of Messages

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# prompt = "What is the capital of France?"  
# print(type(prompt))  # string

# humanMsg = HumanMessage(content="what is the capital of France?")
# print(type(humanMsg))

# print(humanMsg)

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )

# res = llm_model.invoke(humanMsg.content)
# print(res)
# print(type(res))




### ChatPromptTemplate

chatTemplate = ChatPromptTemplate.from_messages([
    ("human", "What is the capital of {countryName}?"),
    ("ai", "The capital of {countryName} is {cityName}"),
    ("human","What is the most popular city of {cityName}")

    # HumanMessage(content="What is the capital of {countryName}?"),
    # AIMessage(content= "The capital of {countryName} is {cityName}"),
    # HumanMessage(content= "What is the most popular city of {cityName}")
])

#res = llm_model.invoke(chatTemplate.invoke({"countryName": "India", "cityName": "Delhi"}))

chain = chatTemplate | llm_model

res = chain.invoke({"countryName": "India", "cityName": "Delhi"})

print(res)


content='Delhi itself is a city—but it\'s actually divided into two main parts:\n\n- **New Delhi**: The planned, northern part of Delhi, which serves as the **national capital territory** and houses government institutions (like the Parliament, Rashtrapati Bhavan, and Supreme Court). It\'s often what people mean when they refer to "Delhi" in an administrative or political context.\n\n- **Old Delhi**: The historic, southern part of Delhi, known for its Mughal-era landmarks like the Red Fort, India Gate, Jama Masjid, and bustling markets like Chandni Chowk.\n\nSo, there isn’t a “most popular city *of* Delhi”—rather, **New Delhi and Old Delhi** are the two iconic regions *within* the larger city of **Delhi** (officially the **National Capital Territory of Delhi**, or NCT).\n\nDelhi is also India’s **second-most populous city** (after Mumbai), and it’s a major cultural, political, and economic hub.\n\nLet me know if you\'d like highlights of Old vs. New Delhi! 😊' additional_kwargs={} respo

### Understanding Tooling in LangChain

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate


# prompt = ChatPromptTemplate.from_messages(    
#     ("human", "What is the weather in {cityName} right now? ")    
#     )
# #prompt = PromptTemplate.from_template(HumanMessage(content ="What is the weather in {cityName} right now? "))
# promptOutput = prompt.invoke({"cityName": "Delhi"})
    #"Also tell me the capital of Australia & "
    #"also tell me the calculated value of 6*7"




@tool
def get_weather_city(city: str) -> str:
    """Get the current weather for a city
    
    Args:
        city: The name of the city for which to get weather information
    
    Returns:
        A string describing the weather in the specified city
    """
    return f"The weather in {city} is sunny."

@tool
def calculate(expression: str) -> str:
    """Calculate the result of a mathematical expression
    
    Args:
        expression: A string representing a mathematical expression
    
    Returns:
        A string representing the result of the calculation
    """
    return str(eval(expression))



def podTestAgent(userprompt: str):
    messages = [HumanMessage(userprompt)]

    tools = [get_weather_city, calculate]
    tools_by_name = {}   # str: StructureTool  {"get_weather_city":  StructuredTool}

    for tool in tools:
        tools_by_name[f"{tool.name}"] = tool   

    llm_model = ChatOllama(
        base_url="https://ollama.com",
        model="qwen3-coder-next:cloud",
        client_kwargs=  {  
            "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
            }
        )

    llm_model_with_tools = llm_model.bind_tools(tools)


    while True:
        res = llm_model_with_tools.invoke(messages)
        #print(res)

        if not res.tool_calls:
            return res.content

        # extract tool_call info
        tools_call = res.tool_calls  # list of tools
        for tool in tools_call:
            tool_name = tool["name"]
            tool_args =tool["args"]
            tool_id = tool["id"]
            tool_result = tools_by_name[tool_name].invoke(tool_args)
            #print(tool_result)

            toolmsg = ToolMessage(content=str(tool_result), tool_call_id=tool_id)
            messages.append(toolmsg)          
   
res = podTestAgent("What is the weather in Delhi right now?  and what is 4 multiply 3? and  what is capital of India?")
print(res)




#print(type(res))


The weather in Delhi is sunny.  
4 multiplied by 3 is 12.  
The capital of India is New Delhi.
